In [16]:
import math
import numpy as np
from dataclasses import dataclass, field

import torch
import torch.nn as nn
import torch.nn.functional as F
import pennylane as qml

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

torch.set_default_dtype(torch.float32)


def bspline_basis_matrix(num_splines: int, degree: int, grid: np.ndarray) -> np.ndarray:
    assert num_splines >= degree + 1
    n = num_splines - 1
    p = degree
    if n - p > 0:
        interior = np.linspace(0.0, 1.0, (n - p) + 2, dtype=float)[1:-1]
    else:
        interior = np.array([], dtype=float)
    knots = np.concatenate([np.zeros(p + 1), interior, np.ones(p + 1)])

    def N(i, r, t):
        if r == 0:
            left = knots[i]
            right = knots[i + 1]
            return np.where(((t >= left) & (t < right)) | ((right == 1.0) & (t == 1.0)), 1.0, 0.0)
        left_den = knots[i + r] - knots[i]
        right_den = knots[i + r + 1] - knots[i + 1]
        left_term = 0.0
        right_term = 0.0
        if left_den > 0:
            left_term = ((t - knots[i]) / left_den) * N(i, r - 1, t)
        if right_den > 0:
            right_term = ((knots[i + r + 1] - t) / right_den) * N(i + 1, r - 1, t)
        return left_term + right_term

    tgrid = np.asarray(grid, dtype=float)
    B = np.vstack([N(i, p, tgrid) for i in range(num_splines)])
    return np.maximum(B, 0.0)


class QCBMState(nn.Module):
    def __init__(self, n_label_qubits: int, n_pos_qubits: int, depth: int = 3, seed: int = 0):
        super().__init__()
        torch.manual_seed(seed)
        self.L = n_label_qubits
        self.P = n_pos_qubits
        self.n_qubits = self.L + self.P
        self.depth = depth
        init = 0.01 * torch.randn(depth, self.n_qubits, 3, dtype=torch.float32)
        self.theta = nn.Parameter(init)
        self.dev = qml.device("default.qubit", wires=self.n_qubits)

        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(weights):
            qml.templates.StronglyEntanglingLayers(weights, wires=list(range(self.n_qubits)))
            return qml.probs(wires=list(range(self.n_qubits)))

        self._qprobs = qnode

    def forward(self):
        return self._qprobs(self.theta).to(torch.float32)

    @torch.no_grad()
    def freeze(self):
        self.theta.requires_grad_(False)


class LabelMixer(nn.Module):
    def __init__(self, qcbm: QCBMState, depth: int = 2, seed: int = 0):
        super().__init__()
        torch.manual_seed(seed)
        self.qcbm = qcbm
        self.L = qcbm.L
        self.P = qcbm.P
        self.n_qubits = qcbm.n_qubits
        self.depth = depth
        init = 0.01 * torch.randn(depth, self.L, 3, dtype=torch.float32)
        self.phi = nn.Parameter(init)
        self.dev = qml.device("default.qubit", wires=self.n_qubits)

        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(weights_qcbm, weights_label):
            qml.templates.StronglyEntanglingLayers(weights_qcbm, wires=list(range(self.n_qubits)))
            if self.L > 0:
                qml.templates.StronglyEntanglingLayers(weights_label, wires=list(range(self.L)))
            return qml.probs(wires=list(range(self.n_qubits)))

        self._qprobs = qnode

    def forward(self):
        return self._qprobs(self.qcbm.theta, self.phi).to(torch.float32)


class QuantumBlock(nn.Module):
    def __init__(self, k_frequencies: int = 4, entangle_depth: int = 1, seed: int = 0):
        super().__init__()
        torch.manual_seed(seed)
        self.K = k_frequencies
        self.depth = entangle_depth
        self.log_omega = nn.Parameter(torch.randn(self.K, dtype=torch.float32) * 0.05)
        self.phase = nn.Parameter(torch.zeros(self.K, dtype=torch.float32))
        self.w_cos = nn.Parameter(torch.randn(self.K, dtype=torch.float32) * 0.1)
        self.w_sin = nn.Parameter(torch.randn(self.K, dtype=torch.float32) * 0.1)
        self.dev = qml.device("default.qubit", wires=self.K)

        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(alpha_vec):
            for k in range(self.K):
                qml.RY(alpha_vec[k], wires=k)
            for _ in range(self.depth):
                for k in range(self.K):
                    qml.CNOT(wires=[k, (k + 1) % self.K])
            z = [qml.expval(qml.PauliZ(k)) for k in range(self.K)]
            x = [qml.expval(qml.PauliX(k)) for k in range(self.K)]
            return z + x

        self._qnode = qnode

    def forward_scalar(self, x01_scalar: torch.Tensor) -> torch.Tensor:
        x01 = torch.clamp(x01_scalar.reshape(()), 0.0, 1.0)
        omega = F.softplus(self.log_omega) + 1e-4
        alpha = omega * (2.0 * math.pi * x01) + self.phase
        outs = self._qnode(alpha.to(torch.float32))
        outs = torch.stack([o if isinstance(o, torch.Tensor) else torch.as_tensor(o, dtype=torch.float32) for o in outs], dim=0).to(torch.float32)
        z = outs[: self.K]
        x = outs[self.K:]
        return (self.w_cos * z).sum() + (self.w_sin * x).sum()

    def forward_batch(self, x01_vec: torch.Tensor) -> torch.Tensor:
        x01_vec = torch.clamp(x01_vec.to(torch.float32), 0.0, 1.0)
        vals = [self.forward_scalar(x01_vec[i]) for i in range(x01_vec.shape[0])]
        return torch.stack(vals, dim=0).to(torch.float32)


class QuKANResidualEdge(nn.Module):
    def __init__(self, mixer: LabelMixer, n_label_qubits: int, n_pos_qubits: int,
                 fourier_k: int = 4, fourier_depth: int = 1, seed: int = 0, w_init=0.5):
        super().__init__()
        self.mixer = mixer
        self.L = n_label_qubits
        self.P = n_pos_qubits
        self.Nlabel = 2 ** self.L
        self.Npos = 2 ** self.P
        self.wf = nn.Parameter(torch.tensor(float(w_init), dtype=torch.float32))
        self.wq = nn.Parameter(torch.tensor(float(w_init), dtype=torch.float32))
        self.qfour = QuantumBlock(k_frequencies=fourier_k, entangle_depth=fourier_depth, seed=seed)

    def batch_forward(self, x_raw: torch.Tensor, x_pos01: torch.Tensor, probs_flat: torch.Tensor) -> torch.Tensor:
        x_pos01 = x_pos01.to(torch.float32)
        probs_flat = probs_flat.to(torch.float32)
        B = x_pos01.shape[0]
        lp = probs_flat.view(self.Nlabel, self.Npos)
        idx = torch.round(torch.clamp(x_pos01, 0.0, 1.0) * (self.Npos - 1)).long()
        idx = torch.clamp(idx, 0, self.Npos - 1)
        p_vals = lp[:, idx].sum(dim=0).to(torch.float32)
        qfr_vals = self.qfour.forward_batch(x_pos01)
        out = (self.wf * p_vals + self.wq * qfr_vals).to(torch.float32)
        return out


@dataclass
class QuKANLayerCfg:
    n_nodes: int = 6
    n_label_qubits: int = 2
    n_pos_qubits: int = 5
    qcbm_depth: int = 3
    label_mixer_depth: int = 2
    fourier_k: int = 4
    fourier_depth: int = 1


class QuKANLayer(nn.Module):
    def __init__(self, cfg: QuKANLayerCfg, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.cfg = cfg
        self.L = cfg.n_label_qubits
        self.P = cfg.n_pos_qubits
        self.qcbm = QCBMState(self.L, self.P, depth=cfg.qcbm_depth, seed=seed)
        self.mixers = nn.ModuleList()
        self.edges = nn.ModuleList()
        self._built = False

    def build(self, input_dim: int, seed: int = 0):
        self.input_dim = input_dim
        torch.manual_seed(seed)
        for m in range(self.cfg.n_nodes):
            for j in range(input_dim):
                mixer = LabelMixer(self.qcbm, depth=self.cfg.label_mixer_depth, seed=seed + 97 * m + j)
                edge = QuKANResidualEdge(
                    mixer,
                    self.L, self.P,
                    fourier_k=self.cfg.fourier_k,
                    fourier_depth=self.cfg.fourier_depth,
                    seed=seed + 991 * m + 13 * j,
                    w_init=0.5
                )
                self.mixers.append(mixer)
                self.edges.append(edge)
        self._built = True
        print(f"[QuKANLayer] built edges: nodes={self.cfg.n_nodes}, in_dim={input_dim}, total_edges={len(self.edges)}")

    def pretrain_qcbm_on_splines(self, degree=2, epochs=200, lr=5e-2, verbose=True):
        num_splines = 2 ** self.L
        Npos = 2 ** self.P
        grid = np.linspace(0.0, 1.0, Npos, dtype=float)
        B = bspline_basis_matrix(num_splines, degree, grid)
        B = B + 1e-8
        B = B / B.sum(axis=1, keepdims=True)
        target = torch.tensor((B / num_splines).reshape(-1), dtype=torch.float32)
        opt = torch.optim.Adam(self.qcbm.parameters(), lr=lr)
        for ep in range(1, epochs + 1):
            opt.zero_grad()
            probs = self.qcbm().to(torch.float32)
            loss = F.mse_loss(probs, target)
            loss.backward()
            opt.step()
            if verbose and (ep % 50 == 0 or ep == 1):
                with torch.no_grad():
                    tv = 0.5 * torch.sum(torch.abs(probs - target)).item()
                print(f"[QCBM pretrain] epoch {ep:03d} | MSE={loss.item():.6f} | TV={tv:.6f}")
        self.qcbm.freeze()

    def forward(self, X_in: torch.Tensor, input_is_01: bool) -> torch.Tensor:
        assert self._built, "Call build(input_dim) first."
        X_in = X_in.to(torch.float32)
        B, D = X_in.shape
        M = self.cfg.n_nodes
        edge_probs = [mix().to(torch.float32) for mix in self.mixers]
        X01_pos = (X_in if input_is_01 else torch.sigmoid(X_in)).to(torch.float32)
        nodes = []
        eidx = 0
        for m in range(M):
            acc = torch.zeros(B, dtype=torch.float32, device=X_in.device)
            for j in range(D):
                probs_flat = edge_probs[eidx]
                edge = self.edges[eidx]
                x_pos = X01_pos[:, j].to(torch.float32)
                out_j = edge.batch_forward(x_pos, x_pos, probs_flat).to(torch.float32)
                acc = acc + out_j
                eidx += 1
            nodes.append(acc)
        nodes = torch.stack(nodes, dim=1).to(torch.float32)
        return nodes


@dataclass
class KANReadoutCfg:
    n_classes: int
    in_dim: int
    fourier_k: int = 3
    fourier_depth: int = 1


class KANReadout(nn.Module):
    def __init__(self, cfg: KANReadoutCfg, seed: int = 0):
        super().__init__()
        torch.manual_seed(seed)
        self.cfg = cfg
        C, M = cfg.n_classes, cfg.in_dim
        self.qfr = nn.ModuleList([
            QuantumBlock(k_frequencies=cfg.fourier_k,
                                entangle_depth=cfg.fourier_depth,
                                seed=seed + 131 * c + m)
            for c in range(C) for m in range(M)
        ])
        self.b = nn.Parameter(torch.zeros(C, dtype=torch.float32))

    def _edge_idx(self, c: int, m: int) -> int:
        return c * self.cfg.in_dim + m

    def forward(self, H: torch.Tensor) -> torch.Tensor:
        H = H.to(torch.float32)
        B, M = H.shape
        C = self.cfg.n_classes
        H01 = torch.sigmoid(H)
        logits = []
        for c in range(C):
            acc_c = torch.zeros(B, dtype=torch.float32, device=H.device)
            for m in range(M):
                qfr = self.qfr[self._edge_idx(c, m)]
                acc_c = acc_c + qfr.forward_batch(H01[:, m])
            logits.append(acc_c + self.b[c])
        return torch.stack(logits, dim=1).to(torch.float32)


@dataclass
class QuKANNetCfg:
    layer1: QuKANLayerCfg = field(default_factory=lambda: QuKANLayerCfg(n_nodes=6, n_label_qubits=2, n_pos_qubits=5, fourier_k=4, fourier_depth=1))
    layer2: QuKANLayerCfg = field(default_factory=lambda: QuKANLayerCfg(n_nodes=6, n_label_qubits=2, n_pos_qubits=5, fourier_k=4, fourier_depth=1))
    n_classes: int = 3


class QuKANNet(nn.Module):
    def __init__(self, cfg: QuKANNetCfg, input_dim: int, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.cfg = cfg
        self.l1 = QuKANLayer(cfg.layer1, seed=seed)
        self.l1.build(input_dim=input_dim, seed=seed)
        self.l2 = QuKANLayer(cfg.layer2, seed=seed + 1)
        self.l2.build(input_dim=cfg.layer1.n_nodes, seed=seed + 1)
        self.readout = KANReadout(
            KANReadoutCfg(
                n_classes=cfg.n_classes,
                in_dim=cfg.layer2.n_nodes,
                fourier_k=3,
                fourier_depth=1
            ),
            seed=seed + 1234
        )

    def pretrain_qcbms(self, degree=2, epochs=200, lr=5e-2, verbose=True):
        print("[Pretrain] Layer 1 QCBM on degree-2 B-splines")
        self.l1.pretrain_qcbm_on_splines(degree=degree, epochs=epochs, lr=lr, verbose=verbose)
        print("[Pretrain] Layer 2 QCBM on degree-2 B-splines")
        self.l2.pretrain_qcbm_on_splines(degree=degree, epochs=epochs, lr=lr, verbose=verbose)

    def forward(self, X01: torch.Tensor) -> torch.Tensor:
        X01 = X01.to(torch.float32)
        h1 = self.l1(X01, input_is_01=True).to(torch.float32)
        h2 = self.l2(h1, input_is_01=False).to(torch.float32)
        return self.readout(h2)


def run_iris(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    iris = load_iris()
    X = iris.data.astype(np.float32)
    y = iris.target.astype(np.int64)
    scaler = MinMaxScaler(feature_range=(0.0, 1.0))
    X01 = scaler.fit_transform(X).astype(np.float32)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X01, y, test_size=0.3, random_state=seed, stratify=y
    )
    X_tr = torch.tensor(X_tr, dtype=torch.float32)
    X_te = torch.tensor(X_te, dtype=torch.float32)
    y_tr = torch.tensor(y_tr, dtype=torch.long)
    y_te = torch.tensor(y_te, dtype=torch.long)
    cfg = QuKANNetCfg(
        layer1=QuKANLayerCfg(n_nodes=6, n_label_qubits=2, n_pos_qubits=5, qcbm_depth=3, label_mixer_depth=2, fourier_k=4, fourier_depth=1),
        layer2=QuKANLayerCfg(n_nodes=6, n_label_qubits=2, n_pos_qubits=5, qcbm_depth=3, label_mixer_depth=2, fourier_k=4, fourier_depth=1),
        n_classes=3,
    )
    model = QuKANNet(cfg, input_dim=4, seed=seed)
    model.pretrain_qcbms(degree=2, epochs=200, lr=5e-2, verbose=True)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=8e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=60)
    ce = nn.CrossEntropyLoss(label_smoothing=0.03)
    print("\nTraining QuKAN on Iris")
    epochs = 60
    B = 16
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(X_tr.shape[0])
        Xb_all, yb_all = X_tr[perm], y_tr[perm]
        tot, corr, loss_sum = 0, 0, 0.0
        for i in range(0, Xb_all.shape[0], B):
            xb = Xb_all[i:i+B]
            yb = yb_all[i:i+B]
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = ce(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            loss_sum += float(loss.item()) * xb.size(0)
            tot += xb.size(0)
            corr += (logits.argmax(1) == yb).sum().item()
        sched.step()
        train_acc = 100.0 * corr / tot
        train_loss = loss_sum / tot
        model.eval()
        with torch.no_grad():
            logits_te = model(X_te)
            val_acc = 100.0 * (logits_te.argmax(1) == y_te).float().mean().item()
        if ep % 2 == 1 or ep >= epochs - 10:
            print(f"Epoch {ep:03d} | Loss={train_loss:.4f} | Train Acc={train_acc:.2f}% | Val Acc={val_acc:.2f}%")
    print("Done.")


if __name__ == "__main__":
    run_iris(seed=0)


[HybridQuKANLayer] built edges: nodes=6, in_dim=4, total_edges=24
[HybridQuKANLayer] built edges: nodes=6, in_dim=6, total_edges=36
[Pretrain] Layer 1 QCBM on degree-2 B-splines
[QCBM pretrain] epoch 001 | MSE=0.007273 | TV=0.955602
[QCBM pretrain] epoch 050 | MSE=0.000061 | TV=0.377527
[QCBM pretrain] epoch 100 | MSE=0.000037 | TV=0.308189
[QCBM pretrain] epoch 150 | MSE=0.000030 | TV=0.276578
[QCBM pretrain] epoch 200 | MSE=0.000028 | TV=0.261863
[Pretrain] Layer 2 QCBM on degree-2 B-splines
[QCBM pretrain] epoch 001 | MSE=0.007276 | TV=0.955730
[QCBM pretrain] epoch 050 | MSE=0.000068 | TV=0.398482
[QCBM pretrain] epoch 100 | MSE=0.000037 | TV=0.306850
[QCBM pretrain] epoch 150 | MSE=0.000029 | TV=0.265133
[QCBM pretrain] epoch 200 | MSE=0.000026 | TV=0.248326

=== Training QuKAN (Hybrid, 2 layers, Quantum Fourier residual + KAN readout) on Iris ===
Epoch 001 | Loss=1.1007 | Train Acc=42.86% | Val Acc=66.67%
Epoch 003 | Loss=0.9679 | Train Acc=70.48% | Val Acc=66.67%
Epoch 005 | Los

KeyboardInterrupt: 

In [17]:
CSV_PATH = r"C:\Users\riakh\Downloads\Social_Network_Ads.csv"
torch.set_default_dtype(torch.float32)

def bspline_basis_matrix(num_splines: int, degree: int, grid: np.ndarray) -> np.ndarray:
    assert num_splines >= degree + 1
    n = num_splines - 1
    p = degree
    if n - p > 0:
        interior = np.linspace(0.0, 1.0, (n - p) + 2, dtype=float)[1:-1]
    else:
        interior = np.array([], dtype=float)
    knots = np.concatenate([np.zeros(p + 1), interior, np.ones(p + 1)])
    def N(i, r, t):
        if r == 0:
            left = knots[i]
            right = knots[i + 1]
            return np.where(((t >= left) & (t < right)) | ((right == 1.0) & (t == 1.0)), 1.0, 0.0)
        left_den = knots[i + r] - knots[i]
        right_den = knots[i + r + 1] - knots[i + 1]
        left_term = 0.0
        right_term = 0.0
        if left_den > 0:
            left_term = ((t - knots[i]) / left_den) * N(i, r - 1, t)
        if right_den > 0:
            right_term = ((knots[i + r + 1] - t) / right_den) * N(i + 1, r - 1, t)
        return left_term + right_term
    tgrid = np.asarray(grid, dtype=float)
    B = np.vstack([N(i, p, tgrid) for i in range(num_splines)])
    return np.maximum(B, 0.0)

class QCBMState(nn.Module):
    def __init__(self, n_label_qubits: int, n_pos_qubits: int, depth: int = 3, seed: int = 0):
        super().__init__()
        torch.manual_seed(seed)
        self.L = n_label_qubits
        self.P = n_pos_qubits
        self.n_qubits = self.L + self.P
        self.depth = depth
        init = 0.01 * torch.randn(depth, self.n_qubits, 3, dtype=torch.float32)
        self.theta = nn.Parameter(init)
        self.dev = qml.device("default.qubit", wires=self.n_qubits)
        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(weights):
            qml.templates.StronglyEntanglingLayers(weights, wires=list(range(self.n_qubits)))
            return qml.probs(wires=list(range(self.n_qubits)))
        self._qprobs = qnode
    def forward(self):
        return self._qprobs(self.theta).to(torch.float32)
    @torch.no_grad()
    def freeze(self):
        self.theta.requires_grad_(False)

class LabelMixer(nn.Module):
    def __init__(self, qcbm: QCBMState, depth: int = 2, seed: int = 0):
        super().__init__()
        torch.manual_seed(seed)
        self.qcbm = qcbm
        self.L = qcbm.L
        self.P = qcbm.P
        self.n_qubits = qcbm.n_qubits
        self.depth = depth
        init = 0.01 * torch.randn(depth, self.L, 3, dtype=torch.float32)
        self.phi = nn.Parameter(init)
        self.dev = qml.device("default.qubit", wires=self.n_qubits)
        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(weights_qcbm, weights_label):
            qml.templates.StronglyEntanglingLayers(weights_qcbm, wires=list(range(self.n_qubits)))
            if self.L > 0:
                qml.templates.StronglyEntanglingLayers(weights_label, wires=list(range(self.L)))
            return qml.probs(wires=list(range(self.n_qubits)))
        self._qprobs = qnode
    def forward(self):
        return self._qprobs(self.qcbm.theta, self.phi).to(torch.float32)

class QuantumBlock(nn.Module):
    def __init__(self, k_frequencies: int = 4, entangle_depth: int = 1, seed: int = 0):
        super().__init__()
        torch.manual_seed(seed)
        self.K = k_frequencies
        self.depth = entangle_depth
        self.log_omega = nn.Parameter(torch.randn(self.K, dtype=torch.float32) * 0.05)
        self.phase = nn.Parameter(torch.zeros(self.K, dtype=torch.float32))
        self.w_cos = nn.Parameter(torch.randn(self.K, dtype=torch.float32) * 0.1)
        self.w_sin = nn.Parameter(torch.randn(self.K, dtype=torch.float32) * 0.1)
        self.dev = qml.device("default.qubit", wires=self.K)
        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(alpha_vec):
            for k in range(self.K):
                qml.RY(alpha_vec[k], wires=k)
            for _ in range(self.depth):
                for k in range(self.K):
                    qml.CNOT(wires=[k, (k + 1) % self.K])
            z = [qml.expval(qml.PauliZ(k)) for k in range(self.K)]
            x = [qml.expval(qml.PauliX(k)) for k in range(self.K)]
            return z + x
        self._qnode = qnode
    def forward_scalar(self, x01_scalar: torch.Tensor) -> torch.Tensor:
        x01 = torch.clamp(x01_scalar.reshape(()), 0.0, 1.0)
        omega = F.softplus(self.log_omega) + 1e-4
        alpha = omega * (2.0 * math.pi * x01) + self.phase
        outs = self._qnode(alpha.to(torch.float32))
        outs = torch.stack([o if isinstance(o, torch.Tensor) else torch.as_tensor(o, dtype=torch.float32)
                            for o in outs], dim=0).to(torch.float32)
        z = outs[: self.K]
        x = outs[self.K:]
        return (self.w_cos * z).sum() + (self.w_sin * x).sum()
    def forward_batch(self, x01_vec: torch.Tensor) -> torch.Tensor:
        x01_vec = torch.clamp(x01_vec.to(torch.float32), 0.0, 1.0)
        vals = [self.forward_scalar(x01_vec[i]) for i in range(x01_vec.shape[0])]
        return torch.stack(vals, dim=0).to(torch.float32)

class QuKANResidualEdge(nn.Module):
    def __init__(self, mixer: LabelMixer, n_label_qubits: int, n_pos_qubits: int,
                 fourier_k: int = 4, fourier_depth: int = 1, seed: int = 0, w_init=0.5):
        super().__init__()
        self.mixer = mixer
        self.L = n_label_qubits
        self.P = n_pos_qubits
        self.Nlabel = 2 ** self.L
        self.Npos = 2 ** self.P
        self.wf = nn.Parameter(torch.tensor(float(w_init), dtype=torch.float32))
        self.wq = nn.Parameter(torch.tensor(float(w_init), dtype=torch.float32))
        self.qfour = QuantumBlock(k_frequencies=fourier_k, entangle_depth=fourier_depth, seed=seed)
    def batch_forward(self, x_pos01: torch.Tensor, probs_flat: torch.Tensor) -> torch.Tensor:
        x_pos01 = x_pos01.to(torch.float32)
        probs_flat = probs_flat.to(torch.float32)
        B = x_pos01.shape[0]
        lp = probs_flat.view(self.Nlabel, self.Npos)
        idx = torch.round(torch.clamp(x_pos01, 0.0, 1.0) * (self.Npos - 1)).long()
        idx = torch.clamp(idx, 0, self.Npos - 1)
        p_vals = lp[:, idx].sum(dim=0).to(torch.float32)
        qfr_vals = self.qfour.forward_batch(x_pos01)
        out = (self.wf * p_vals + self.wq * qfr_vals).to(torch.float32)
        return out

@dataclass
class QuKANLayerCfg:
    n_nodes: int = 6
    n_label_qubits: int = 2
    n_pos_qubits: int = 5
    qcbm_depth: int = 3
    label_mixer_depth: int = 2
    fourier_k: int = 4
    fourier_depth: int = 1

class QuKANLayer(nn.Module):
    def __init__(self, cfg: QuKANLayerCfg, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.cfg = cfg
        self.L = cfg.n_label_qubits
        self.P = cfg.n_pos_qubits
        self.qcbm = QCBMState(self.L, self.P, depth=cfg.qcbm_depth, seed=seed)
        self.mixers = nn.ModuleList()
        self.edges = nn.ModuleList()
        self._built = False
    def build(self, input_dim: int, seed: int = 0):
        self.input_dim = input_dim
        torch.manual_seed(seed)
        for m in range(self.cfg.n_nodes):
            for j in range(input_dim):
                mixer = LabelMixer(self.qcbm, depth=self.cfg.label_mixer_depth, seed=seed + 97 * m + j)
                edge = QuKANResidualEdge(
                    mixer,
                    self.L, self.P,
                    fourier_k=self.cfg.fourier_k,
                    fourier_depth=self.cfg.fourier_depth,
                    seed=seed + 991 * m + 13 * j,
                    w_init=0.5
                )
                self.mixers.append(mixer)
                self.edges.append(edge)
        self._built = True
        print(f"[QuKANLayer] built edges: nodes={self.cfg.n_nodes}, in_dim={input_dim}, total_edges={len(self.edges)}")
    def pretrain_qcbm_on_splines(self, degree=2, epochs=200, lr=5e-2, verbose=True):
        num_splines = 2 ** self.L
        Npos = 2 ** self.P
        grid = np.linspace(0.0, 1.0, Npos, dtype=float)
        B = bspline_basis_matrix(num_splines, degree, grid)
        B = B + 1e-8
        B = B / B.sum(axis=1, keepdims=True)
        target = torch.tensor((B / num_splines).reshape(-1), dtype=torch.float32)
        opt = torch.optim.Adam(self.qcbm.parameters(), lr=lr)
        for ep in range(1, epochs + 1):
            opt.zero_grad()
            probs = self.qcbm().to(torch.float32)
            loss = F.mse_loss(probs, target)
            loss.backward()
            opt.step()
            if verbose and (ep % 50 == 0 or ep == 1):
                with torch.no_grad():
                    tv = 0.5 * torch.sum(torch.abs(probs - target)).item()
                print(f"[QCBM pretrain] epoch {ep:03d} | MSE={loss.item():.6f} | TV={tv:.6f}")
        self.qcbm.freeze()
    def forward(self, X_in: torch.Tensor, input_is_01: bool) -> torch.Tensor:
        assert self._built, "Call build(input_dim) first."
        X_in = X_in.to(torch.float32)
        B, D = X_in.shape
        M = self.cfg.n_nodes
        edge_probs = [mix().to(torch.float32) for mix in self.mixers]
        X01_pos = (X_in if input_is_01 else torch.sigmoid(X_in)).to(torch.float32)
        nodes = []
        eidx = 0
        for m in range(M):
            acc = torch.zeros(B, dtype=torch.float32, device=X_in.device)
            for j in range(D):
                probs_flat = edge_probs[eidx]
                edge = self.edges[eidx]
                x_pos = X01_pos[:, j].to(torch.float32)
                out_j = edge.batch_forward(x_pos, probs_flat).to(torch.float32)
                acc = acc + out_j
                eidx += 1
            nodes.append(acc)
        nodes = torch.stack(nodes, dim=1).to(torch.float32)
        return nodes

@dataclass
class KANReadoutCfg:
    n_classes: int
    in_dim: int
    fourier_k: int = 3
    fourier_depth: int = 1

class KANReadout(nn.Module):
    def __init__(self, cfg: KANReadoutCfg, seed: int = 0):
        super().__init__()
        torch.manual_seed(seed)
        self.cfg = cfg
        C, M = cfg.n_classes, cfg.in_dim
        self.qfr = nn.ModuleList([
            QuantumBlock(k_frequencies=cfg.fourier_k,
                                entangle_depth=cfg.fourier_depth,
                                seed=seed + 131 * c + m)
            for c in range(C) for m in range(M)
        ])
        self.b = nn.Parameter(torch.zeros(C, dtype=torch.float32))
    def _edge_idx(self, c: int, m: int) -> int:
        return c * self.cfg.in_dim + m
    def forward(self, H: torch.Tensor) -> torch.Tensor:
        H = H.to(torch.float32)
        B, M = H.shape
        C = self.cfg.n_classes
        H01 = torch.sigmoid(H)
        logits = []
        for c in range(C):
            acc_c = torch.zeros(B, dtype=torch.float32, device=H.device)
            for m in range(M):
                qfr = self.qfr[self._edge_idx(c, m)]
                acc_c = acc_c + qfr.forward_batch(H01[:, m])
            logits.append(acc_c + self.b[c])
        return torch.stack(logits, dim=1).to(torch.float32)

@dataclass
class QuKANNetCfg:
    layer1: QuKANLayerCfg = field(default_factory=lambda: QuKANLayerCfg(n_nodes=6, n_label_qubits=2, n_pos_qubits=5, fourier_k=4, fourier_depth=1))
    layer2: QuKANLayerCfg = field(default_factory=lambda: QuKANLayerCfg(n_nodes=6, n_label_qubits=2, n_pos_qubits=5, fourier_k=4, fourier_depth=1))
    n_classes: int = 2

class QuKANNet(nn.Module):
    def __init__(self, cfg: QuKANNetCfg, input_dim: int, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.cfg = cfg
        self.l1 = QuKANLayer(cfg.layer1, seed=seed)
        self.l1.build(input_dim=input_dim, seed=seed)
        self.l2 = QuKANLayer(cfg.layer2, seed=seed + 1)
        self.l2.build(input_dim=cfg.layer1.n_nodes, seed=seed + 1)
        self.readout = KANReadout(
            KANReadoutCfg(
                n_classes=cfg.n_classes,
                in_dim=cfg.layer2.n_nodes,
                fourier_k=3,
                fourier_depth=1
            ),
            seed=seed + 1234
        )
    def pretrain_qcbms(self, degree=2, epochs=200, lr=5e-2, verbose=True):
        print("[Pretrain] Layer 1 QCBM on degree-2 B-splines")
        self.l1.pretrain_qcbm_on_splines(degree=degree, epochs=epochs, lr=lr, verbose=verbose)
        print("[Pretrain] Layer 2 QCBM on degree-2 B-splines")
        self.l2.pretrain_qcbm_on_splines(degree=degree, epochs=epochs, lr=lr, verbose=verbose)
    def forward(self, X01: torch.Tensor) -> torch.Tensor:
        X01 = X01.to(torch.float32)
        h1 = self.l1(X01, input_is_01=True).to(torch.float32)
        h2 = self.l2(h1,  input_is_01=False).to(torch.float32)
        return self.readout(h2)

def run_social(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    assert os.path.exists(CSV_PATH), f"CSV not found: {CSV_PATH}"
    df = pd.read_csv(CSV_PATH)
    cols = [c.lower() for c in df.columns]
    col_map = {c.lower(): c for c in df.columns}
    needed = ["age", "estimatedsalary", "purchased"]
    for k in needed:
        assert k in cols, f"Column '{k}' not found in CSV. Found columns: {df.columns.tolist()}"
    X_np = df[[col_map["age"], col_map["estimatedsalary"]]].values.astype(np.float32)
    y_np = df[col_map["purchased"]].values.astype(np.int64)
    scaler = MinMaxScaler(feature_range=(0.0, 1.0))
    X01 = scaler.fit_transform(X_np).astype(np.float32)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X01, y_np, test_size=0.3, random_state=seed, stratify=y_np
    )
    X_tr = torch.tensor(X_tr, dtype=torch.float32)
    X_te = torch.tensor(X_te, dtype=torch.float32)
    y_tr = torch.tensor(y_tr, dtype=torch.long)
    y_te = torch.tensor(y_te, dtype=torch.long)
    cfg = QuKANNetCfg(
        layer1=QuKANLayerCfg(n_nodes=6, n_label_qubits=2, n_pos_qubits=5,
                             qcbm_depth=3, label_mixer_depth=2, fourier_k=4, fourier_depth=1),
        layer2=QuKANLayerCfg(n_nodes=6, n_label_qubits=2, n_pos_qubits=5,
                             qcbm_depth=3, label_mixer_depth=2, fourier_k=4, fourier_depth=1),
        n_classes=2,
    )
    model = QuKANNet(cfg, input_dim=2, seed=seed)
    model.pretrain_qcbms(degree=2, epochs=200, lr=5e-2, verbose=True)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=8e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=60)
    ce = nn.CrossEntropyLoss(label_smoothing=0.03)
    print("\nTraining QuKAN on Social_Network_Ads")
    epochs = 60
    B = 32
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(X_tr.shape[0])
        Xb_all, yb_all = X_tr[perm], y_tr[perm]
        tot, corr, loss_sum = 0, 0, 0.0
        for i in range(0, Xb_all.shape[0], B):
            xb = Xb_all[i:i+B]
            yb = yb_all[i:i+B]
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = ce(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            loss_sum += float(loss.item()) * xb.size(0)
            tot += xb.size(0)
            corr += (logits.argmax(1) == yb).sum().item()
        sched.step()
        train_acc = 100.0 * corr / tot
        train_loss = loss_sum / tot
        model.eval()
        with torch.no_grad():
            logits_te = model(X_te)
            val_acc = 100.0 * (logits_te.argmax(1) == y_te).float().mean().item()
        if ep % 2 == 1 or ep >= epochs - 10:
            print(f"Epoch {ep:03d} | Loss={train_loss:.4f} | Train Acc={train_acc:.2f}% | Val Acc={val_acc:.2f}%")
    print("Done.")
    with torch.no_grad():
        pred = model(X_te).argmax(1).cpu().numpy()
        acc = (pred == y_te.cpu().numpy()).mean() * 100
    print(f"\nFinal Test Accuracy: {acc:.2f}%")

if __name__ == "__main__":
    run_social(seed=0)


[HybridQuKANLayer] built edges: nodes=6, in_dim=2, total_edges=12
[HybridQuKANLayer] built edges: nodes=6, in_dim=6, total_edges=36
[Pretrain] Layer 1 QCBM on degree-2 B-splines
[QCBM pretrain] epoch 001 | MSE=0.007273 | TV=0.955602
[QCBM pretrain] epoch 050 | MSE=0.000061 | TV=0.377527
[QCBM pretrain] epoch 100 | MSE=0.000037 | TV=0.308189
[QCBM pretrain] epoch 150 | MSE=0.000030 | TV=0.276578
[QCBM pretrain] epoch 200 | MSE=0.000028 | TV=0.261863
[Pretrain] Layer 2 QCBM on degree-2 B-splines
[QCBM pretrain] epoch 001 | MSE=0.007276 | TV=0.955730
[QCBM pretrain] epoch 050 | MSE=0.000068 | TV=0.398482
[QCBM pretrain] epoch 100 | MSE=0.000037 | TV=0.306850
[QCBM pretrain] epoch 150 | MSE=0.000029 | TV=0.265133
[QCBM pretrain] epoch 200 | MSE=0.000026 | TV=0.248326

=== Training QuKAN (Hybrid, 2 layers, QFR + KAN readout) on Social_Network_Ads ===
Epoch 001 | Loss=0.6937 | Train Acc=52.50% | Val Acc=64.17%
Epoch 003 | Loss=0.6446 | Train Acc=64.29% | Val Acc=64.17%
Epoch 005 | Loss=0.578

KeyboardInterrupt: 

In [4]:
torch.set_default_dtype(torch.float32)
CSV_PATH = r"C:\Users\riakh\Downloads\archive\Titanic-Dataset.csv"

def bspline_basis_matrix(num_splines: int, degree: int, grid: np.ndarray) -> np.ndarray:
    assert num_splines >= degree + 1
    n = num_splines - 1
    p = degree
    if n - p > 0:
        interior = np.linspace(0.0, 1.0, (n - p) + 2, dtype=float)[1:-1]
    else:
        interior = np.array([], dtype=float)
    knots = np.concatenate([np.zeros(p + 1), interior, np.ones(p + 1)])
    def N(i, r, t):
        if r == 0:
            left, right = knots[i], knots[i + 1]
            return np.where(((t >= left) & (t < right)) | ((right == 1.0) & (t == 1.0)), 1.0, 0.0)
        left_den = knots[i + r] - knots[i]
        right_den = knots[i + r + 1] - knots[i + 1]
        left_term = ((t - knots[i]) / left_den) * N(i, r - 1, t) if left_den > 0 else 0
        right_term = ((knots[i + r + 1] - t) / right_den) * N(i + 1, r - 1, t) if right_den > 0 else 0
        return left_term + right_term
    tgrid = np.asarray(grid, dtype=float)
    return np.vstack([N(i, p, tgrid) for i in range(num_splines)])

class QCBMState(nn.Module):
    def __init__(self, n_label_qubits, n_pos_qubits, depth=3, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.L, self.P = n_label_qubits, n_pos_qubits
        self.n_qubits = self.L + self.P
        self.theta = nn.Parameter(0.01 * torch.randn(depth, self.n_qubits, 3, dtype=torch.float32))
        self.dev = qml.device("default.qubit", wires=self.n_qubits)
        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(weights):
            qml.templates.StronglyEntanglingLayers(weights, wires=range(self.n_qubits))
            return qml.probs(wires=range(self.n_qubits))
        self._qprobs = qnode
    def forward(self):
        return self._qprobs(self.theta).to(torch.float32)
    def freeze(self):
        self.theta.requires_grad_(False)

class LabelMixer(nn.Module):
    def __init__(self, qcbm: QCBMState, depth=1, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.qcbm = qcbm
        self.L, self.P = qcbm.L, qcbm.P
        self.phi = nn.Parameter(0.01 * torch.randn(depth, self.L, 3, dtype=torch.float32))
        self.dev = qml.device("default.qubit", wires=self.L + self.P)
        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(weights_qcbm, weights_label):
            qml.templates.StronglyEntanglingLayers(weights_qcbm, wires=range(self.L + self.P))
            if self.L > 0:
                qml.templates.StronglyEntanglingLayers(weights_label, wires=range(self.L))
            return qml.probs(wires=range(self.L + self.P))
        self._qprobs = qnode
    def forward(self):
        return self._qprobs(self.qcbm.theta, self.phi).to(torch.float32)

class QuantumBlock(nn.Module):
    def __init__(self, k_frequencies=3, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.K = k_frequencies
        self.log_omega = nn.Parameter(torch.randn(self.K) * 0.05)
        self.phase = nn.Parameter(torch.zeros(self.K))
        self.w_cos = nn.Parameter(torch.randn(self.K) * 0.1)
        self.w_sin = nn.Parameter(torch.randn(self.K) * 0.1)
    def forward_batch(self, x01_vec):
        x01_vec = torch.clamp(x01_vec, 0, 1)
        omega = F.softplus(self.log_omega) + 1e-4
        vals = []
        for val in x01_vec:
            alpha = omega * (2*math.pi*val) + self.phase
            z = torch.cos(alpha)
            x = torch.sin(alpha)
            vals.append((self.w_cos*z).sum() + (self.w_sin*x).sum())
        return torch.stack(vals)

class QuKANResidualEdge(nn.Module):
    def __init__(self, mixer, n_label_qubits, n_pos_qubits, fourier_k=3, seed=0, w_init=0.5):
        super().__init__()
        self.mixer = mixer
        self.L, self.P = n_label_qubits, n_pos_qubits
        self.Nlabel, self.Npos = 2**self.L, 2**self.P
        self.wf = nn.Parameter(torch.tensor(float(w_init)))
        self.wq = nn.Parameter(torch.tensor(float(w_init)))
        self.qfour = QuantumBlock(fourier_k, seed=seed)
    def batch_forward(self, x_pos01, probs_flat):
        lp = probs_flat.view(self.Nlabel, self.Npos)
        idx = torch.round(torch.clamp(x_pos01,0,1)*(self.Npos-1)).long()
        idx = torch.clamp(idx, 0, self.Npos-1)
        p_vals = lp[:,idx].sum(0)
        qfr_vals = self.qfour.forward_batch(x_pos01)
        return self.wf*p_vals + self.wq*qfr_vals

@dataclass
class QuKANLayerCfg:
    n_nodes: int = 6
    n_label_qubits: int = 2
    n_pos_qubits: int = 5
    qcbm_depth: int = 3
    label_mixer_depth: int = 1
    fourier_k: int = 3
    mixers_trainable: bool = False

class QuKANLayer(nn.Module):
    def __init__(self, cfg: QuKANLayerCfg, seed=0):
        super().__init__()
        self.cfg = cfg
        self.qcbm = QCBMState(cfg.n_label_qubits, cfg.n_pos_qubits, cfg.qcbm_depth, seed)
        self.mixers, self.edges = nn.ModuleList(), nn.ModuleList()
        self._built=False
        self._train_mixers = cfg.mixers_trainable
    def build(self, input_dim, seed=0):
        for m in range(self.cfg.n_nodes):
            for j in range(input_dim):
                mixer = LabelMixer(self.qcbm, self.cfg.label_mixer_depth, seed+97*m+j)
                edge = QuKANResidualEdge(mixer, self.cfg.n_label_qubits, self.cfg.n_pos_qubits,
                                         self.cfg.fourier_k, seed=seed+991*m+13*j)
                self.mixers.append(mixer); self.edges.append(edge)
        self._built=True
        print(f"[QuKANLayer] built edges: {self.cfg.n_nodes} nodes × {input_dim} inputs = {len(self.edges)} edges")
    def pretrain_qcbm_on_splines(self, degree=2, epochs=80, lr=5e-2, verbose=True):
        num_spl, Npos = 2**self.cfg.n_label_qubits, 2**self.cfg.n_pos_qubits
        grid = np.linspace(0,1,Npos)
        B = bspline_basis_matrix(num_spl, degree, grid)
        B = (B+1e-8)/B.sum(1,keepdims=True)
        target = torch.tensor((B/num_spl).reshape(-1), dtype=torch.float32)
        opt=torch.optim.Adam(self.qcbm.parameters(), lr=lr)
        for ep in range(epochs):
            opt.zero_grad(); probs=self.qcbm()
            loss=F.mse_loss(probs, target); loss.backward(); opt.step()
            if verbose and (ep%20==0 or ep==epochs-1):
                tv=0.5*torch.sum(torch.abs(probs-target)).item()
                print(f"[QCBM pretrain] {ep:03d} | MSE={loss.item():.6f} | TV={tv:.6f}")
        self.qcbm.freeze()
        print(">> QCBM frozen.")
    def forward(self,X, input_is_01=True):
        X01 = (X if input_is_01 else torch.sigmoid(X))
        if self._train_mixers:
            edge_probs=[mix() for mix in self.mixers]
        else:
            with torch.no_grad():
                edge_probs=[mix() for mix in self.mixers]
        nodes=[]; eidx=0
        for m in range(self.cfg.n_nodes):
            acc=torch.zeros(X.shape[0], dtype=torch.float32)
            for j in range(X.shape[1]):
                out=self.edges[eidx].batch_forward(X01[:,j], edge_probs[eidx])
                acc=acc+out; eidx+=1
            nodes.append(acc)
        return torch.stack(nodes,1)

@dataclass
class KANReadoutCfg:
    n_classes:int; in_dim:int; fourier_k:int=3

class KANReadout(nn.Module):
    def __init__(self,cfg:KANReadoutCfg,seed=0):
        super().__init__()
        self.cfg=cfg; C,M=cfg.n_classes,cfg.in_dim
        self.qfr=nn.ModuleList([QuantumBlock(cfg.fourier_k,seed+131*c+m)
                                for c in range(C) for m in range(M)])
        self.b=nn.Parameter(torch.zeros(C))
    def _idx(self,c,m): return c*self.cfg.in_dim+m
    def forward(self,H):
        H01=torch.sigmoid(H); logits=[]
        for c in range(self.cfg.n_classes):
            acc=torch.zeros(H.shape[0], dtype=torch.float32)
            for m in range(H.shape[1]):
                acc=acc+self.qfr[self._idx(c,m)].forward_batch(H01[:,m])
            logits.append(acc+self.b[c])
        return torch.stack(logits,1)

@dataclass
class QuKANNetCfg:
    layer1:QuKANLayerCfg=field(default_factory=QuKANLayerCfg)
    layer2:QuKANLayerCfg=field(default_factory=QuKANLayerCfg)
    n_classes:int=2

class QuKANNet(nn.Module):
    def __init__(self,cfg,input_dim,seed=0):
        super().__init__()
        self.l1=QuKANLayer(cfg.layer1,seed); self.l1.build(input_dim,seed)
        self.l2=QuKANLayer(cfg.layer2,seed+1); self.l2.build(cfg.layer1.n_nodes,seed+1)
        self.readout=KANReadout(KANReadoutCfg(cfg.n_classes,cfg.layer2.n_nodes),seed+123)
    def pretrain_qcbms(self,degree=2,epochs=80,lr=5e-2):
        print("\n[Pretrain] Layer 1 QCBM"); self.l1.pretrain_qcbm_on_splines(degree,epochs,lr)
        print("\n[Pretrain] Layer 2 QCBM"); self.l2.pretrain_qcbm_on_splines(degree,epochs,lr)
    def forward(self,X):
        h1=self.l1(X,True); h2=self.l2(h1,False); return self.readout(h2)

def _first_present(cols_map, names):
    for n in names:
        if n in cols_map:
            return cols_map[n]
    return None

def load_titanic_features(csv_path: str):
    assert os.path.exists(csv_path), f"CSV not found: {csv_path}"
    df = pd.read_csv(csv_path)
    cols_map = {c.lower(): c for c in df.columns}
    survived = _first_present(cols_map, ["survived"])
    pclass   = _first_present(cols_map, ["pclass","p class","p_class"])
    sex      = _first_present(cols_map, ["sex","gender"])
    age      = _first_present(cols_map, ["age"])
    sibsp    = _first_present(cols_map, ["sibsp","siblings/spouses aboard","siblingsaboard","siblings_spouses_aboard"])
    parch    = _first_present(cols_map, ["parch","parents/children aboard","parentschildrenaboard","parents_children_aboard"])
    fare     = _first_present(cols_map, ["fare"])
    embarked = _first_present(cols_map, ["embarked","port of embarkation","emb"])
    for k,v in {"survived":survived,"pclass":pclass,"sex":sex,"age":age,
                "sibsp":sibsp,"parch":parch,"fare":fare,"embarked":embarked}.items():
        if v is None:
            raise ValueError(f"Could not find required column alias for '{k}'. Found columns: {list(df.columns)}")
    sub = df[[survived, pclass, sex, age, sibsp, parch, fare, embarked]].copy()
    sub.columns = [c.lower() for c in sub.columns]
    sub["sex"] = sub["sex"].astype(str).str.lower().map({"male": 0, "m": 0, "female": 1, "f": 1})
    sub["embarked"] = sub["embarked"].astype(str).str.upper().map({"S": "S", "C": "C", "Q": "Q"})
    sub["age"] = pd.to_numeric(sub["age"], errors="coerce")
    sub["fare"] = pd.to_numeric(sub["fare"], errors="coerce")
    sub["age"] = sub["age"].fillna(sub["age"].median())
    sub["fare"] = sub["fare"].fillna(sub["fare"].median())
    sub["sex"] = sub["sex"].fillna(0)
    sub["embarked"] = sub["embarked"].fillna("S")
    one_hot_emb = pd.get_dummies(sub["embarked"], prefix="emb", drop_first=False)
    one_hot_cls = pd.get_dummies(sub["pclass"].astype(int), prefix="pcls", drop_first=False)
    X = pd.concat(
        [one_hot_cls,
         sub[["sex", "age", "sibsp", "parch", "fare"]].reset_index(drop=True),
         one_hot_emb.reset_index(drop=True)],axis=1).astype(np.float32)
    y = sub["survived"].astype(np.int64).to_numpy()
    scaler = MinMaxScaler((0.0, 1.0))
    X01 = scaler.fit_transform(X.to_numpy().astype(np.float32)).astype(np.float32)
    return X01, y

def run_titanic(seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    X, y = load_titanic_features(CSV_PATH)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    X_tr, X_te = torch.tensor(X_tr, dtype=torch.float32), torch.tensor(X_te, dtype=torch.float32)
    y_tr, y_te = torch.tensor(y_tr, dtype=torch.long),  torch.tensor(y_te, dtype=torch.long)
    input_dim = X_tr.shape[1]
    print(f"Input dim: {input_dim}")
    cfg = QuKANNetCfg(
        layer1=QuKANLayerCfg(n_nodes=6, n_label_qubits=2, n_pos_qubits=5, label_mixer_depth=1, fourier_k=3, mixers_trainable=False),
        layer2=QuKANLayerCfg(n_nodes=6, n_label_qubits=2, n_pos_qubits=5, label_mixer_depth=1, fourier_k=3, mixers_trainable=False),
        n_classes=2
    )
    model = QuKANNet(cfg, input_dim=input_dim, seed=seed)
    model.pretrain_qcbms()
    opt = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=8e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40)
    ce = nn.CrossEntropyLoss(label_smoothing=0.03)
    print("\nTraining QuKAN on Titanic")
    for ep in range(1, 41):
        model.train()
        perm = torch.randperm(X_tr.shape[0])
        Xb_all, yb_all = X_tr[perm], y_tr[perm]
        loss_sum, tot, corr = 0.0, 0, 0
        for i in range(0, Xb_all.shape[0], 64):
            xb, yb = Xb_all[i:i+64], yb_all[i:i+64]
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = ce(logits, yb)
            loss.backward()
            opt.step()
            loss_sum += float(loss.item()) * xb.size(0)
            tot += xb.size(0)
            corr += (logits.argmax(1) == yb).sum().item()
        sched.step()
        train_acc = 100.0 * corr / tot
        val_acc = (model(X_te).argmax(1) == y_te).float().mean().item() * 100.0
        print(f"Epoch {ep:03d} | Train Acc={train_acc:.2f}% | Val Acc={val_acc:.2f}%")

if __name__ == "__main__":
    run_titanic(0)


Input dim: 11
[HybridQuKANLayer] built edges: 6 nodes × 11 inputs = 66 edges
[HybridQuKANLayer] built edges: 6 nodes × 6 inputs = 36 edges

[Pretrain] Layer 1 QCBM
[QCBM pretrain] 000 | MSE=0.007273 | TV=0.955602
[QCBM pretrain] 020 | MSE=0.000122 | TV=0.517162
[QCBM pretrain] 040 | MSE=0.000075 | TV=0.411801
[QCBM pretrain] 060 | MSE=0.000052 | TV=0.350588
[QCBM pretrain] 079 | MSE=0.000043 | TV=0.327553
>> QCBM frozen.

[Pretrain] Layer 2 QCBM
[QCBM pretrain] 000 | MSE=0.007276 | TV=0.955730
[QCBM pretrain] 020 | MSE=0.000119 | TV=0.506432
[QCBM pretrain] 040 | MSE=0.000074 | TV=0.419522
[QCBM pretrain] 060 | MSE=0.000059 | TV=0.372766
[QCBM pretrain] 079 | MSE=0.000047 | TV=0.344233
>> QCBM frozen.

=== Training QuKAN on Titanic ===
Epoch 001 | Train Acc=61.64% | Val Acc=61.57%
Epoch 002 | Train Acc=61.64% | Val Acc=61.57%
Epoch 003 | Train Acc=62.92% | Val Acc=67.91%
Epoch 004 | Train Acc=66.93% | Val Acc=73.13%
Epoch 005 | Train Acc=75.92% | Val Acc=79.10%
Epoch 006 | Train Acc=77

In [8]:
def bspline_basis_matrix(num_splines: int, degree: int, grid: np.ndarray) -> np.ndarray:
    assert num_splines >= degree + 1
    n = num_splines - 1
    p = degree
    if n - p > 0:
        interior = np.linspace(0.0, 1.0, (n - p) + 2, dtype=float)[1:-1]
    else:
        interior = np.array([], dtype=float)
    knots = np.concatenate([np.zeros(p + 1), interior, np.ones(p + 1)])
    def N(i, r, t):
        if r == 0:
            left, right = knots[i], knots[i + 1]
            return np.where(((t >= left) & (t < right)) | ((right == 1.0) & (t == 1.0)), 1.0, 0.0)
        left_den = knots[i + r] - knots[i]
        right_den = knots[i + r + 1] - knots[i + 1]
        left_term = ((t - knots[i]) / left_den) * N(i, r - 1, t) if left_den > 0 else 0
        right_term = ((knots[i + r + 1] - t) / right_den) * N(i + 1, r - 1, t) if right_den > 0 else 0
        return left_term + right_term
    tgrid = np.asarray(grid, dtype=float)
    return np.vstack([N(i, p, tgrid) for i in range(num_splines)])

class QCBMState(nn.Module):
    def __init__(self, n_label_qubits, n_pos_qubits, depth=3, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.L, self.P = n_label_qubits, n_pos_qubits
        self.n_qubits = self.L + self.P
        self.theta = nn.Parameter(0.01 * torch.randn(depth, self.n_qubits, 3, dtype=torch.float32))
        self.dev = qml.device("default.qubit", wires=self.n_qubits)
        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(weights):
            qml.templates.StronglyEntanglingLayers(weights, wires=range(self.n_qubits))
            return qml.probs(wires=range(self.n_qubits))
        self._qprobs = qnode
    def forward(self):
        return self._qprobs(self.theta).to(torch.float32)
    def freeze(self):
        self.theta.requires_grad_(False)

class LabelMixer(nn.Module):
    def __init__(self, qcbm: QCBMState, depth=1, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.qcbm = qcbm
        self.L, self.P = qcbm.L, qcbm.P
        self.phi = nn.Parameter(0.01 * torch.randn(depth, self.L, 3, dtype=torch.float32))
        self.dev = qml.device("default.qubit", wires=self.L + self.P)
        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(weights_qcbm, weights_label):
            qml.templates.StronglyEntanglingLayers(weights_qcbm, wires=range(self.L + self.P))
            if self.L > 0:
                qml.templates.StronglyEntanglingLayers(weights_label, wires=range(self.L))
            return qml.probs(wires=range(self.L + self.P))
        self._qprobs = qnode
    def forward(self):
        return self._qprobs(self.qcbm.theta, self.phi).to(torch.float32)

class QuantumBlock(nn.Module):
    def __init__(self, k_frequencies=3, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.K = k_frequencies
        self.log_omega = nn.Parameter(torch.randn(self.K) * 0.05)
        self.phase = nn.Parameter(torch.zeros(self.K))
        self.w_cos = nn.Parameter(torch.randn(self.K) * 0.1)
        self.w_sin = nn.Parameter(torch.randn(self.K) * 0.1)
    def forward_batch(self, x01_vec):
        x01_vec = torch.clamp(x01_vec, 0, 1)
        omega = F.softplus(self.log_omega) + 1e-4
        vals = []
        for val in x01_vec:
            alpha = omega * (2*math.pi*val) + self.phase
            z = torch.cos(alpha)
            x = torch.sin(alpha)
            vals.append((self.w_cos*z).sum() + (self.w_sin*x).sum())
        return torch.stack(vals)

class QuKANResidualEdge(nn.Module):
    def __init__(self, mixer, n_label_qubits, n_pos_qubits, fourier_k=3, seed=0, w_init=0.5):
        super().__init__()
        self.mixer = mixer
        self.L, self.P = n_label_qubits, n_pos_qubits
        self.Nlabel, self.Npos = 2**self.L, 2**self.P
        self.wf = nn.Parameter(torch.tensor(float(w_init)))
        self.wq = nn.Parameter(torch.tensor(float(w_init)))
        self.qfour = QuantumBlock(fourier_k, seed=seed)
    def batch_forward(self, x_pos01, probs_flat):
        lp = probs_flat.view(self.Nlabel, self.Npos)
        idx = torch.round(torch.clamp(x_pos01,0,1)*(self.Npos-1)).long()
        idx = torch.clamp(idx, 0, self.Npos-1)
        p_vals = lp[:,idx].sum(0)
        qfr_vals = self.qfour.forward_batch(x_pos01)
        return self.wf*p_vals + self.wq*qfr_vals

@dataclass
class QuKANLayerCfg:
    n_nodes: int = 4
    n_label_qubits: int = 2
    n_pos_qubits: int = 6
    qcbm_depth: int = 3
    label_mixer_depth: int = 1
    fourier_k: int = 3
    mixers_trainable: bool = False

class QuKANLayer(nn.Module):
    def __init__(self, cfg: QuKANLayerCfg, seed=0):
        super().__init__()
        self.cfg = cfg
        self.qcbm = QCBMState(cfg.n_label_qubits, cfg.n_pos_qubits, cfg.qcbm_depth, seed)
        self.mixers, self.edges = nn.ModuleList(), nn.ModuleList()
        self._built=False
        self._train_mixers = cfg.mixers_trainable
    def build(self, input_dim, seed=0):
        for m in range(self.cfg.n_nodes):
            for j in range(input_dim):
                mixer = LabelMixer(self.qcbm, self.cfg.label_mixer_depth, seed+97*m+j)
                edge = QuKANResidualEdge(mixer, self.cfg.n_label_qubits, self.cfg.n_pos_qubits,
                                         self.cfg.fourier_k, seed=seed+991*m+13*j)
                self.mixers.append(mixer); self.edges.append(edge)
        self._built=True
        print(f"[QuKANLayer] built edges: {self.cfg.n_nodes} nodes × {input_dim} inputs = {len(self.edges)} edges")
    def pretrain_qcbm_on_splines(self, degree=2, epochs=80, lr=5e-2, verbose=True):
        num_spl, Npos = 2**self.cfg.n_label_qubits, 2**self.cfg.n_pos_qubits
        grid = np.linspace(0,1,Npos)
        B = bspline_basis_matrix(num_spl, degree, grid)
        B = (B+1e-8)/B.sum(1,keepdims=True)
        target = torch.tensor((B/num_spl).reshape(-1), dtype=torch.float32)
        opt=torch.optim.Adam(self.qcbm.parameters(), lr=lr)
        for ep in range(epochs):
            opt.zero_grad(); probs=self.qcbm()
            loss=F.mse_loss(probs, target); loss.backward(); opt.step()
            if verbose and (ep%20==0 or ep==epochs-1):
                tv=0.5*torch.sum(torch.abs(probs-target)).item()
                print(f"[QCBM pretrain] {ep:03d} | MSE={loss.item():.6f} | TV={tv:.6f}")
        self.qcbm.freeze()
        print("QCBM frozen.")
    def forward(self,X, input_is_01=True):
        X01 = (X if input_is_01 else torch.sigmoid(X))
        if self._train_mixers:
            edge_probs=[mix() for mix in self.mixers]
        else:
            with torch.no_grad():
                edge_probs=[mix() for mix in self.mixers]
        nodes=[]; eidx=0
        for m in range(self.cfg.n_nodes):
            acc=torch.zeros(X.shape[0], dtype=torch.float32)
            for j in range(X.shape[1]):
                out=self.edges[eidx].batch_forward(X01[:,j], edge_probs[eidx])
                acc=acc+out; eidx+=1
            nodes.append(acc)
        return torch.stack(nodes,1)

@dataclass
class KANReadoutCfg:
    n_classes:int; in_dim:int; fourier_k:int=3

class KANReadout(nn.Module):
    def __init__(self,cfg:KANReadoutCfg,seed=0):
        super().__init__()
        self.cfg=cfg; C,M=cfg.n_classes,cfg.in_dim
        self.qfr=nn.ModuleList([QuantumBlock(cfg.fourier_k,seed+131*c+m)
                                for c in range(C) for m in range(M)])
        self.b=nn.Parameter(torch.zeros(C))
    def _idx(self,c,m): return c*self.cfg.in_dim+m
    def forward(self,H):
        H01=torch.sigmoid(H); logits=[]
        for c in range(self.cfg.n_classes):
            acc=torch.zeros(H.shape[0], dtype=torch.float32)
            for m in range(H.shape[1]):
                acc=acc+self.qfr[self._idx(c,m)].forward_batch(H01[:,m])
            logits.append(acc+self.b[c])
        return torch.stack(logits,1)

@dataclass
class QuKANNetCfg:
    layer1:QuKANLayerCfg=field(default_factory=QuKANLayerCfg)
    layer2:QuKANLayerCfg=field(default_factory=lambda: QuKANLayerCfg(n_pos_qubits=6))
    n_classes:int=10

class QuKANNet(nn.Module):
    def __init__(self,cfg,input_dim,seed=0):
        super().__init__()
        self.l1=QuKANLayer(cfg.layer1,seed); self.l1.build(input_dim,seed)
        self.l2=QuKANLayer(cfg.layer2,seed+1); self.l2.build(cfg.layer1.n_nodes,seed+1)
        self.readout=KANReadout(KANReadoutCfg(cfg.n_classes,cfg.layer2.n_nodes),seed+123)
    def pretrain_qcbms(self,degree=2,epochs=80,lr=5e-2):
        print("\n[Pretrain] Layer 1 QCBM"); self.l1.pretrain_qcbm_on_splines(degree,epochs,lr)
        print("\n[Pretrain] Layer 2 QCBM"); self.l2.pretrain_qcbm_on_splines(degree,epochs,lr)
    def forward(self,X):
        h1=self.l1(X,True); h2=self.l2(h1,False); return self.readout(h2)

def run_digits(seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    digits = load_digits()
    X, y = digits.data.astype(np.float32), digits.target.astype(np.int64)
    X, y = X[:1000], y[:1000]
    X = MinMaxScaler((0,1)).fit_transform(X).astype(np.float32)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    X_tr, X_te = torch.tensor(X_tr), torch.tensor(X_te)
    y_tr, y_te = torch.tensor(y_tr), torch.tensor(y_te)
    model = QuKANNet(QuKANNetCfg(), input_dim=64, seed=seed)
    model.pretrain_qcbms()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=8e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40)
    ce = nn.CrossEntropyLoss(label_smoothing=0.05)
    print("\nTraining QuKAN on Digits Dataset (1000 samples)")
    for ep in range(1, 41):
        model.train()
        perm = torch.randperm(X_tr.shape[0])
        Xb_all, yb_all = X_tr[perm], y_tr[perm]
        loss_sum, tot, corr = 0.0, 0, 0
        for i in range(0, Xb_all.shape[0], 64):
            xb, yb = Xb_all[i:i+64], yb_all[i:i+64]
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = ce(logits, yb)
            loss.backward()
            opt.step()
            loss_sum += float(loss.item()) * xb.size(0)
            tot += xb.size(0)
            corr += (logits.argmax(1) == yb).sum().item()
        sched.step()
        train_acc = 100.0 * corr / tot
        val_acc = (model(X_te).argmax(1) == y_te).float().mean().item() * 100.0
        print(f"Epoch {ep:03d} | Train Acc={train_acc:.2f}% | Val Acc={val_acc:.2f}%")

if __name__ == "__main__":
    run_digits(0)


[HybridQuKANLayer] built edges: 4 nodes × 64 inputs = 256 edges
[HybridQuKANLayer] built edges: 4 nodes × 4 inputs = 16 edges

[Pretrain] Layer 1 QCBM
[QCBM pretrain] 000 | MSE=0.003764 | TV=0.976948
[QCBM pretrain] 020 | MSE=0.000040 | TV=0.549158
[QCBM pretrain] 040 | MSE=0.000027 | TV=0.492827
[QCBM pretrain] 060 | MSE=0.000024 | TV=0.469451
[QCBM pretrain] 079 | MSE=0.000021 | TV=0.438543
>> QCBM frozen.

[Pretrain] Layer 2 QCBM
[QCBM pretrain] 000 | MSE=0.003764 | TV=0.976846
[QCBM pretrain] 020 | MSE=0.000039 | TV=0.556429
[QCBM pretrain] 040 | MSE=0.000029 | TV=0.503609
[QCBM pretrain] 060 | MSE=0.000023 | TV=0.457479
[QCBM pretrain] 079 | MSE=0.000020 | TV=0.429788
>> QCBM frozen.

=== Training QuKAN on Digits Dataset (1000 samples) ===
Epoch 001 | Train Acc=9.86% | Val Acc=10.00%
Epoch 002 | Train Acc=11.71% | Val Acc=13.67%
Epoch 003 | Train Acc=23.29% | Val Acc=29.33%
Epoch 004 | Train Acc=31.29% | Val Acc=31.67%
Epoch 005 | Train Acc=34.57% | Val Acc=35.00%
Epoch 006 | Trai

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pennylane as qml
import math
from dataclasses import dataclass, field
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

torch.set_default_dtype(torch.float32)

class QCBMState(nn.Module):
    def __init__(self, n_label_qubits, n_pos_qubits, depth=3, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.L, self.P = n_label_qubits, n_pos_qubits
        self.n_qubits = self.L + self.P
        self.theta = nn.Parameter(0.01 * torch.randn(depth, self.n_qubits, 3, dtype=torch.float32))
        self.dev = qml.device("default.qubit", wires=self.n_qubits)
        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(weights):
            qml.templates.StronglyEntanglingLayers(weights, wires=range(self.n_qubits))
            return qml.probs(wires=range(self.n_qubits))
        self._qprobs = qnode
    def forward(self):
        return self._qprobs(self.theta).to(torch.float32)

class LabelMixer(nn.Module):
    def __init__(self, qcbm: QCBMState, depth=2, seed=0):
        super().__init__() 
        torch.manual_seed(seed)
        self.qcbm = qcbm
        self.L, self.P = qcbm.L, qcbm.P
        self.phi = nn.Parameter(0.01 * torch.randn(depth, self.L, 3, dtype=torch.float32))
        self.dev = qml.device("default.qubit", wires=self.L + self.P)
        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(weights_qcbm, weights_label):
            qml.templates.StronglyEntanglingLayers(weights_qcbm, wires=range(self.L + self.P))
            if self.L > 0:
                qml.templates.StronglyEntanglingLayers(weights_label, wires=range(self.L))
            return qml.probs(wires=range(self.L + self.P))
        self._qprobs = qnode
    def forward(self):
        return self._qprobs(self.qcbm.theta, self.phi).to(torch.float32)

class QuantumBlock(nn.Module):
    def __init__(self, k_frequencies=4, entangle_depth=1, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.K = k_frequencies
        self.log_omega = nn.Parameter(torch.randn(self.K) * 0.05)
        self.phase = nn.Parameter(torch.zeros(self.K))
        self.w_cos = nn.Parameter(torch.randn(self.K) * 0.1)
        self.w_sin = nn.Parameter(torch.randn(self.K) * 0.1)
        self.dev = qml.device("default.qubit", wires=self.K)
        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(alpha_vec):
            for k in range(self.K):
                qml.RY(alpha_vec[k], wires=k)
            for _ in range(entangle_depth):
                for k in range(self.K):
                    qml.CNOT([k, (k + 1) % self.K])
            z = [qml.expval(qml.PauliZ(k)) for k in range(self.K)]
            x = [qml.expval(qml.PauliX(k)) for k in range(self.K)]
            return z + x
        self._qnode = qnode
    def forward_batch(self, x01_vec: torch.Tensor):
        x01_vec = torch.clamp(x01_vec, 0, 1)
        omega = F.softplus(self.log_omega) + 1e-4
        vals = []
        for val in x01_vec:
            alpha = omega * (2 * math.pi * val) + self.phase
            outs = self._qnode(alpha.to(torch.float32))
            outs = torch.stack([torch.as_tensor(o, dtype=torch.float32) for o in outs])
            vals.append((self.w_cos * outs[:self.K]).sum() + (self.w_sin * outs[self.K:]).sum())
        return torch.stack(vals)

class QuKANResidualEdge(nn.Module):
    def __init__(self, mixer, n_label_qubits, n_pos_qubits, fourier_k=4, fourier_depth=1, seed=0, w_init=0.5):
        super().__init__()
        self.mixer = mixer
        self.L, self.P = n_label_qubits, n_pos_qubits
        self.Nlabel, self.Npos = 2 ** self.L, 2 ** self.P
        self.wf = nn.Parameter(torch.tensor(float(w_init)))
        self.wq = nn.Parameter(torch.tensor(float(w_init)))
        self.qfour = QuantumBlock(fourier_k, fourier_depth, seed=seed)
    def batch_forward(self, x_pos01: torch.Tensor, probs_flat: torch.Tensor):
        lp = probs_flat.view(self.Nlabel, self.Npos)
        idx = torch.round(torch.clamp(x_pos01, 0, 1) * (self.Npos - 1)).long()
        idx = torch.clamp(idx, 0, self.Npos - 1)
        p_vals = lp[:, idx].sum(0)
        qfr_vals = self.qfour.forward_batch(x_pos01)
        return self.wf * p_vals + self.wq * qfr_vals

@dataclass
class QuKANLayerCfg:
    n_nodes: int = 5
    n_label_qubits: int = 2
    n_pos_qubits: int = 6
    qcbm_depth: int = 3
    label_mixer_depth: int = 2
    fourier_k: int = 4
    fourier_depth: int = 1

class QuKANLayer(nn.Module):
    def __init__(self, cfg: QuKANLayerCfg, seed=0):
        super().__init__()
        self.cfg = cfg
        self.qcbm = QCBMState(cfg.n_label_qubits, cfg.n_pos_qubits, cfg.qcbm_depth, seed)
        self.mixers, self.edges = nn.ModuleList(), nn.ModuleList()
    def build(self, input_dim, seed=0):
        for m in range(self.cfg.n_nodes):
            for j in range(input_dim):
                mixer = LabelMixer(self.qcbm, self.cfg.label_mixer_depth, seed + 97 * m + j)
                edge = QuKANResidualEdge(
                    mixer, self.cfg.n_label_qubits, self.cfg.n_pos_qubits,
                    self.cfg.fourier_k, self.cfg.fourier_depth, seed + 991 * m + 13 * j
                )
                self.mixers.append(mixer)
                self.edges.append(edge)
    def forward(self, X):
        X01 = torch.sigmoid(X)
        edge_probs = [mix() for mix in self.mixers]
        nodes = []
        eidx = 0
        for m in range(self.cfg.n_nodes):
            acc = torch.zeros(X.shape[0], dtype=torch.float32, device=X.device)
            for j in range(X.shape[1]):
                out = self.edges[eidx].batch_forward(X01[:, j], edge_probs[eidx])
                acc = acc + out
                eidx += 1
            nodes.append(acc)
        return torch.stack(nodes, 1)

@dataclass
class KANReadoutCfg:
    n_classes: int
    in_dim: int
    fourier_k: int = 3
    fourier_depth: int = 1

class KANReadout(nn.Module):
    def __init__(self, cfg: KANReadoutCfg, seed=0):
        super().__init__()
        self.cfg = cfg
        C, M = cfg.n_classes, cfg.in_dim
        self.qfr = nn.ModuleList([
            QuantumBlock(cfg.fourier_k, cfg.fourier_depth, seed + 131 * c + m)
            for c in range(C) for m in range(M)
        ])
        self.b = nn.Parameter(torch.zeros(C))
    def _idx(self, c, m):
        return c * self.cfg.in_dim + m
    def forward(self, H):
        H01 = torch.sigmoid(H)
        logits = []
        for c in range(self.cfg.n_classes):
            acc = torch.zeros(H.shape[0], dtype=torch.float32, device=H.device)
            for m in range(H.shape[1]):
                acc = acc + self.qfr[self._idx(c, m)].forward_batch(H01[:, m])
            logits.append(acc + self.b[c])
        return torch.stack(logits, 1)

@dataclass
class QuKANNetCfg:
    layer1: QuKANLayerCfg = field(default_factory=QuKANLayerCfg)
    layer2: QuKANLayerCfg = field(default_factory=QuKANLayerCfg)
    n_classes: int = 2 

class QuKANNet(nn.Module):
    def __init__(self, cfg, input_dim, seed=0):
        super().__init__()
        self.l1 = QuKANLayer(cfg.layer1, seed);   self.l1.build(input_dim, seed)
        self.l2 = QuKANLayer(cfg.layer2, seed+1); self.l2.build(cfg.layer1.n_nodes, seed+1)
        self.readout = KANReadout(KANReadoutCfg(cfg.n_classes, cfg.layer2.n_nodes), seed+123)
    def forward(self, X):
        h1 = self.l1(X)
        h2 = self.l2(h1)
        return self.readout(h2)

def load_higgs_csv_first_n(csv_path: str, n_samples: int):
    data = np.loadtxt(csv_path, delimiter=",", max_rows=n_samples)
    y = data[:, 0].astype(np.int64)
    X = data[:, 1:29].astype(np.float32)
    scaler = MinMaxScaler((0, 1))
    X = scaler.fit_transform(X).astype(np.float32)
    return X, y

def run_higgs(csv_path: str, n_samples: int = 20000, epochs: int = 20, batch_size: int = 128, seed: int = 0):
    torch.manual_seed(seed); np.random.seed(seed)
    X, y = load_higgs_csv_first_n(csv_path, n_samples)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    X_tr = torch.tensor(X_tr, dtype=torch.float32)
    X_te = torch.tensor(X_te, dtype=torch.float32)
    y_tr = torch.tensor(y_tr, dtype=torch.long)
    y_te = torch.tensor(y_te, dtype=torch.long)
    input_dim = X_tr.shape[1] 
    model = QuKANNet(QuKANNetCfg(), input_dim=input_dim, seed=seed)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    ce = nn.CrossEntropyLoss(label_smoothing=0.05)
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(X_tr.shape[0])
        xb_all, yb_all = X_tr[perm], y_tr[perm]
        tot, corr = 0, 0
        epoch_loss = 0.0
        for i in range(0, xb_all.shape[0], batch_size):
            xb = xb_all[i:i+batch_size]
            yb = yb_all[i:i+batch_size]
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = ce(logits, yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * xb.size(0)
            tot += xb.size(0)
            corr += (logits.argmax(1) == yb).sum().item()
        train_acc = 100.0 * corr / tot
        with torch.no_grad():
            val_logits = model(X_te)
            val_acc = (val_logits.argmax(1) == y_te).float().mean().item() * 100.0
        avg_loss = epoch_loss / tot
        print(f"Epoch {ep:03d} | Loss={avg_loss:.4f} | Train Acc={train_acc:.2f}% | Val Acc={val_acc:.2f}%")

if __name__ == "__main__":
    CSV_PATH = r"C:\Users\riakh\Downloads\archive (26)\HIGGS.csv"
    run_higgs(CSV_PATH, n_samples=2000, epochs=20, batch_size=128, seed=0)


[Data] Loading first 2000 rows from: C:\Users\riakh\Downloads\archive (26)\HIGGS.csv
[QuKANNet] Initializing network...
[HybridQuKANLayer] Building with 28 inputs...
[HybridQuKANLayer] Built edges: 5 nodes × 28 inputs = 140 edges
[HybridQuKANLayer] Building with 5 inputs...
[HybridQuKANLayer] Built edges: 5 nodes × 5 inputs = 25 edges
[QuKANNet] Build complete.

=== Training QuKAN on HIGGS (28 features) ===
Epoch 001 | Loss=0.6926 | Train Acc=53.75% | Val Acc=53.75%
Epoch 002 | Loss=0.6905 | Train Acc=53.75% | Val Acc=53.75%
Epoch 003 | Loss=0.6895 | Train Acc=53.75% | Val Acc=53.75%
Epoch 004 | Loss=0.6897 | Train Acc=53.94% | Val Acc=53.75%
Epoch 005 | Loss=0.6877 | Train Acc=53.75% | Val Acc=54.00%
Epoch 006 | Loss=0.6871 | Train Acc=57.62% | Val Acc=53.75%
Epoch 007 | Loss=0.6850 | Train Acc=54.75% | Val Acc=55.50%
Epoch 008 | Loss=0.6819 | Train Acc=56.94% | Val Acc=55.50%
Epoch 009 | Loss=0.6759 | Train Acc=58.25% | Val Acc=56.75%
Epoch 010 | Loss=0.6719 | Train Acc=59.62% | Val 

In [9]:
# ============================================================
# Fully Quantum KAN (QuKAN) – Regression on Nonlinear Functions
# With build logs and QCBM pretraining
# ============================================================

import math
import numpy as np
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
import pennylane as qml
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float32)


# ---------------------------
# B-spline basis (for QCBM pretraining)
# ---------------------------
def bspline_basis_matrix(num_splines: int, degree: int, grid: np.ndarray) -> np.ndarray:
    assert num_splines >= degree + 1
    n = num_splines - 1
    p = degree
    if n - p > 0:
        interior = np.linspace(0.0, 1.0, (n - p) + 2, dtype=float)[1:-1]
    else:
        interior = np.array([], dtype=float)
    knots = np.concatenate([np.zeros(p + 1), interior, np.ones(p + 1)])

    def N(i, r, t):
        if r == 0:
            left, right = knots[i], knots[i + 1]
            return np.where(((t >= left) & (t < right)) | ((right == 1.0) & (t == 1.0)), 1.0, 0.0)
        left_den = knots[i + r] - knots[i]
        right_den = knots[i + r + 1] - knots[i + 1]
        left_term = ((t - knots[i]) / left_den) * N(i, r - 1, t) if left_den > 0 else 0
        right_term = ((knots[i + r + 1] - t) / right_den) * N(i + 1, r - 1, t) if right_den > 0 else 0
        return left_term + right_term

    tgrid = np.asarray(grid, dtype=float)
    return np.vstack([N(i, p, tgrid) for i in range(num_splines)])


# ---------------------------
# QCBM over label+position
# ---------------------------
class QCBMState(nn.Module):
    def __init__(self, n_label_qubits: int, n_pos_qubits: int, depth: int = 3, seed: int = 0):
        super().__init__()
        torch.manual_seed(seed)
        self.L, self.P = n_label_qubits, n_pos_qubits
        self.n_qubits = self.L + self.P
        self.theta = nn.Parameter(0.01 * torch.randn(depth, self.n_qubits, 3).float())

        self.dev = qml.device("default.qubit", wires=self.n_qubits)

        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(weights):
            qml.templates.StronglyEntanglingLayers(weights, wires=range(self.n_qubits))
            return qml.probs(wires=range(self.n_qubits))

        self._qprobs = qnode

    def forward(self):
        return self._qprobs(self.theta.float()).to(torch.float32)


class LabelMixer(nn.Module):
    def __init__(self, qcbm: QCBMState, depth=1, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.qcbm = qcbm
        self.L, self.P = qcbm.L, qcbm.P
        self.phi = nn.Parameter(0.01 * torch.randn(depth, self.L, 3).float())

        self.dev = qml.device("default.qubit", wires=self.L + self.P)

        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(weights_qcbm, weights_label):
            qml.templates.StronglyEntanglingLayers(weights_qcbm, wires=range(self.L + self.P))
            if self.L > 0:
                qml.templates.StronglyEntanglingLayers(weights_label, wires=range(self.L))
            return qml.probs(wires=range(self.L + self.P))

        self._qprobs = qnode

    def forward(self):
        return self._qprobs(self.qcbm.theta.float(), self.phi.float()).to(torch.float32)


class QuantumBlock(nn.Module):
    def __init__(self, k_frequencies=3, depth=1, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.K = k_frequencies
        self.log_omega = nn.Parameter(torch.randn(self.K).float() * 0.05)
        self.phase = nn.Parameter(torch.zeros(self.K).float())
        self.w_cos = nn.Parameter(torch.randn(self.K).float() * 0.1)
        self.w_sin = nn.Parameter(torch.randn(self.K).float() * 0.1)

        self.dev = qml.device("default.qubit", wires=self.K)

        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(alpha_vec):
            for k in range(self.K):
                qml.RY(alpha_vec[k], wires=k)
            for k in range(self.K - 1):
                qml.CNOT(wires=[k, k + 1])
            z = [qml.expval(qml.PauliZ(k)) for k in range(self.K)]
            x = [qml.expval(qml.PauliX(k)) for k in range(self.K)]
            return z + x

        self._qnode = qnode

    def forward_scalar(self, x01_scalar: torch.Tensor) -> torch.Tensor:
        x01 = torch.clamp(x01_scalar.reshape(()).float(), 0.0, 1.0)
        omega = F.softplus(self.log_omega.float()) + 1e-4
        alpha = omega * (2 * math.pi * x01) + self.phase.float()
        outs = self._qnode(alpha.float())
        outs = torch.stack([torch.as_tensor(o, dtype=torch.float32) for o in outs], 0)
        z, x = outs[:self.K], outs[self.K:]
        return (self.w_cos.float() * z).sum() + (self.w_sin.float() * x).sum()

    def forward_batch(self, x01_vec: torch.Tensor) -> torch.Tensor:
        return torch.stack([self.forward_scalar(val) for val in x01_vec.float()], 0)


class QuKANResidualEdge(nn.Module):
    def __init__(self, mixer: LabelMixer, n_label_qubits, n_pos_qubits, k=3):
        super().__init__()
        self.mixer = mixer
        self.Nlabel, self.Npos = 2 ** n_label_qubits, 2 ** n_pos_qubits
        self.wf = nn.Parameter(torch.tensor(0.5).float())
        self.wq = nn.Parameter(torch.tensor(0.5).float())
        self.qfour = QuantumBlock(k)

    def batch_forward(self, x_pos01, probs_flat):
        lp = probs_flat.view(self.Nlabel, self.Npos)
        idx = torch.round(torch.clamp(x_pos01.float(), 0, 1) * (self.Npos - 1)).long()
        idx = torch.clamp(idx, 0, self.Npos - 1)
        p_vals = lp[:, idx].sum(0).float()
        qfr_vals = self.qfour.forward_batch(x_pos01.float())
        return self.wf * p_vals + self.wq * qfr_vals


class QuKANRegressor(nn.Module):
    def __init__(self, input_dim=1, hidden_nodes=6, seed=0):
        super().__init__()
        self.qcbm = QCBMState(2, 5, depth=3, seed=seed)
        self.mixers, self.edges = nn.ModuleList(), nn.ModuleList()
        for m in range(hidden_nodes):
            for j in range(input_dim):
                mixer = LabelMixer(self.qcbm, depth=1, seed=seed + 97 * m + j)
                edge = QuKANResidualEdge(mixer, 2, 5, k=3)
                self.mixers.append(mixer)
                self.edges.append(edge)

        print(f"[QuKANRegressor] built edges: {hidden_nodes} nodes × {input_dim} inputs = {len(self.edges)} edges")

        self.readout = QuantumBlock(k_frequencies=3, seed=seed + 123)

    def pretrain_qcbm(self, degree=2, epochs=50, lr=5e-2):
        num_spl, Npos = 2 ** self.qcbm.L, 2 ** self.qcbm.P
        grid = np.linspace(0, 1, Npos)
        B = np.maximum(bspline_basis_matrix(num_spl, degree, grid), 0.0)
        B = (B + 1e-8) / B.sum(1, keepdims=True)
        target = torch.tensor((B / num_spl).reshape(-1), dtype=torch.float32)

        opt = torch.optim.Adam(self.qcbm.parameters(), lr=lr)
        for ep in range(epochs):
            opt.zero_grad()
            probs = self.qcbm()
            loss = F.mse_loss(probs, target)
            loss.backward()
            opt.step()
            if ep % 10 == 0 or ep == epochs - 1:
                tv = 0.5 * torch.sum(torch.abs(probs - target)).item()
                print(f"[QCBM pretrain] {ep:03d} | MSE={loss.item():.6f} | TV={tv:.6f}")

        self.qcbm.theta.requires_grad_(False)
        print(">> QCBM frozen.")

    def forward(self, X):
        X01 = torch.sigmoid(X.float())
        edge_probs = [mix().float() for mix in self.mixers]
        nodes, eidx = [], 0
        for m in range(len(self.mixers) // X.shape[1]):
            acc = torch.zeros(X.shape[0]).float()
            for j in range(X.shape[1]):
                out = self.edges[eidx].batch_forward(X01[:, j], edge_probs[eidx])
                acc = acc + out
                eidx += 1
            nodes.append(acc)
        H = torch.stack(nodes, 1).float()
        return self.readout.forward_batch(H.mean(1))


# ---------------------------
# Target functions
# ---------------------------
def f_func(x): return torch.tanh(10*x + 0.5 + F.relu(x**2) * 10)
def g_func(x): return torch.sin(x) + torch.cos(5*x) * torch.exp(-x**2) + F.relu(x - 0.5)
def h_func(x): return torch.sigmoid(3*x) + F.relu(torch.sin(2*x) + x**3)
def k_func(x): return torch.tanh(5*x - 2) + 3 * F.relu(torch.cos(x**2))
def m_func(x): return F.softplus(x**2 - 1) + torch.tanh(4*x + 0.1)
def n_func(x): return torch.exp(-x**2 + 0.3*x) + F.relu(torch.tanh(2*x - 1))

FUNCTION_MAP = {
    "f_func": f_func,
    "g_func": g_func,
    "h_func": h_func,
    "k_func": k_func,
    "m_func": m_func,
    "n_func": n_func,
}


def train_one_function(name, func, epochs=100, batch=64, seed=0):
    x = torch.linspace(-1, 1, 500).unsqueeze(1).float()
    y = func(x).float()
    X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

    model = QuKANRegressor(input_dim=1, hidden_nodes=6, seed=seed)
    model.pretrain_qcbm()

    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=8e-4)
    mse = nn.MSELoss()

    train_losses, test_losses = [], []

    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        pred = model(X_train)
        loss = mse(pred, y_train.squeeze())
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        with torch.no_grad():
            model.eval()
            test_loss = mse(model(X_test), y_test.squeeze())

        train_losses.append(loss.item())
        test_losses.append(test_loss.item())

        if (ep + 1) % 5 == 0 or ep == epochs - 1:
            print(f"[{name}] Epoch {ep+1} | Train Loss={loss.item():.5f} | Test Loss={test_loss.item():.5f}")

    # ---------- Plotting ----------
    model.eval()
    with torch.no_grad():
        preds = model(X_test).cpu().numpy()
        true = y_test.cpu().numpy()
        x_plot = X_test.cpu().numpy().squeeze()
        sort_idx = x_plot.argsort()

    plt.figure(figsize=(12, 5))

    # Prediction vs Ground Truth
    plt.subplot(1, 2, 1)
    plt.plot(x_plot[sort_idx], true[sort_idx], label='Ground Truth', color='blue')
    plt.plot(x_plot[sort_idx], preds[sort_idx], '--', label='Prediction', color='red')
    plt.title(f"{name} – Prediction vs Ground Truth")
    plt.xlabel("Input x")
    plt.ylabel("f(x)")
    plt.legend()
    plt.grid(True)

    # Loss curves\
    
    plt.subplot(1, 2, 2)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.title(f"{name} – Loss over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(f"{name}_results.png")  # single file with both plots
    plt.close()


def main():
    for name, fn in FUNCTION_MAP.items():
        print(f"\n=== Training QuKAN Regressor on {name} ===")
        train_one_function(name, fn, epochs=500, batch=64, seed=0)



if __name__ == "__main__":
    main()



=== Training QuKAN Regressor on f_func ===
[QuKANRegressor] built edges: 6 nodes × 1 inputs = 6 edges
[QCBM pretrain] 000 | MSE=0.007273 | TV=0.955602
[QCBM pretrain] 010 | MSE=0.000574 | TV=0.650723
[QCBM pretrain] 020 | MSE=0.000122 | TV=0.517162
[QCBM pretrain] 030 | MSE=0.000094 | TV=0.466575
[QCBM pretrain] 040 | MSE=0.000075 | TV=0.411801
[QCBM pretrain] 049 | MSE=0.000061 | TV=0.377527
>> QCBM frozen.
[f_func] Epoch 5 | Train Loss=0.80472 | Test Loss=0.87720
[f_func] Epoch 10 | Train Loss=0.79135 | Test Loss=0.85996
[f_func] Epoch 15 | Train Loss=0.78032 | Test Loss=0.84498
[f_func] Epoch 20 | Train Loss=0.77089 | Test Loss=0.83136
[f_func] Epoch 25 | Train Loss=0.76133 | Test Loss=0.81692
[f_func] Epoch 30 | Train Loss=0.74736 | Test Loss=0.79742
[f_func] Epoch 35 | Train Loss=0.72494 | Test Loss=0.76910
[f_func] Epoch 40 | Train Loss=0.69029 | Test Loss=0.72731
[f_func] Epoch 45 | Train Loss=0.64046 | Test Loss=0.67059
[f_func] Epoch 50 | Train Loss=0.57680 | Test Loss=0.6015